# Market and News Data Collection

Enter a stock ticker when prompted. The notebook validates the symbol against Yahoo Finance, resolves its company name, and persists both `STOCK_TICKER` and `COMPANY_NAME` in the project-root `.env` file so downstream notebooks can reuse them.

It then collects historical market data and company-related news. API credentials are read from the runtime environment and are never stored in this notebook.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
os.chdir('/content/drive/MyDrive/AAI-520-Project')

print(f"current dir: {Path.cwd()}")

Mounted at /content/drive
current dir: /content/drive/MyDrive/AAI-520-Project


In [3]:
import os
import re
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv, set_key

project_root = Path.cwd()

env_path = project_root / ".env"
load_dotenv(env_path, override=True)
print(f"Project root: {project_root}")
print(f"Environment file: {env_path}")


Project root: /content/drive/MyDrive/AAI-520-Project
Environment file: /content/drive/MyDrive/AAI-520-Project/.env


In [4]:
ticker_input = input(
    "Enter a valid stock ticker symbol (for example, AAPL or MSFT): "
).strip().upper()

# Permit common exchange/ticker punctuation while rejecting natural-language prompts.
ticker_pattern = re.compile(r"^[A-Z][A-Z0-9]*(?:[.-][A-Z0-9]+)*$")
if not ticker_input:
    raise ValueError(
        "A stock ticker is required. Enter a ticker symbol such as AAPL or MSFT; "
        "do not enter a company name or a research question."
    )
if len(ticker_input) > 15 or not ticker_pattern.fullmatch(ticker_input):
    raise ValueError(
        f"Invalid stock ticker input: {ticker_input!r}. Enter a ticker symbol "
        "such as AAPL, MSFT, or BRK-B—not a company name or a question."
    )

stock_ticker = ticker_input
data_period = os.getenv("DATA_PERIOD", "1y").strip()
if not data_period:
    raise ValueError("DATA_PERIOD is empty. Set it in the project .env file.")

print(f"Requested ticker: {stock_ticker}")
print(f"Market-data period: {data_period}")


Enter a valid stock ticker symbol (for example, AAPL or MSFT): SNDK
Requested ticker: SNDK
Market-data period: 1y


In [5]:
stock = yf.Ticker(stock_ticker)

try:
    stock_df = stock.history(period=data_period).reset_index()
except Exception as exc:
    raise ValueError(
        f"Could not validate ticker {stock_ticker!r} with Yahoo Finance. "
        "Check the symbol and try again."
    ) from exc

if stock_df.empty:
    raise ValueError(
        f"{stock_ticker!r} did not return market data from Yahoo Finance. "
        "The symbol may be invalid, unsupported, or unavailable. "
        "Restart this cell and enter a valid stock ticker."
    )

try:
    stock_info = stock.info or {}
except Exception as exc:
    raise RuntimeError(
        f"Market data was found for {stock_ticker}, but Yahoo Finance company "
        "metadata could not be retrieved. Please retry later."
    ) from exc

company_name = (
    stock_info.get("longName")
    or stock_info.get("shortName")
    or stock_info.get("displayName")
)
if not isinstance(company_name, str) or not company_name.strip():
    raise ValueError(
        f"Yahoo Finance returned market data for {stock_ticker}, but no company "
        "name could be resolved. Please verify the ticker or retry later."
    )
company_name = company_name.strip()

# Persist the selected company and ticker so later notebooks can load them.
env_path.touch(exist_ok=True)
set_key(str(env_path), "STOCK_TICKER", stock_ticker, quote_mode="never")
set_key(str(env_path), "COMPANY_NAME", company_name, quote_mode="auto")
os.environ["STOCK_TICKER"] = stock_ticker
os.environ["COMPANY_NAME"] = company_name

display(stock_df.head())
print(f"Validated ticker: {stock_ticker}")
print(f"Resolved company name: {company_name}")
print(f"Market rows: {len(stock_df):,}")
print("Persisted STOCK_TICKER and COMPANY_NAME to the project .env file.")


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2025-10-15 00:00:00-04:00,134.750000,144.550003,130.500000,144.300003,8971100,0.0,0.0
1,2025-10-16 00:00:00-04:00,146.000000,149.089996,141.699997,144.270004,6289600,0.0,0.0
2,2025-10-17 00:00:00-04:00,139.220001,150.820007,136.289993,140.160004,8279800,0.0,0.0
3,2025-10-20 00:00:00-04:00,145.979996,152.630005,145.500000,148.039993,6626100,0.0,0.0
4,2025-10-21 00:00:00-04:00,145.800003,150.160004,142.759995,149.289993,4177900,0.0,0.0


Validated ticker: SNDK
Resolved company name: Sandisk Corporation
Market rows: 252
Persisted STOCK_TICKER and COMPANY_NAME to the project .env file.


In [6]:
news_api_key = os.getenv("NEWS_API_KEY")

if not news_api_key:
    raise RuntimeError(
        "NEWS_API_KEY is missing. Add NEWS_API_KEY=... to the project "
        ".env file, then restart the notebook kernel."
    )

response = requests.get(
    "https://newsapi.org/v2/everything",
    params={
        "q": stock_ticker,
        "apiKey": news_api_key,
        "language": "en",
        "pageSize": 50,
        "sortBy": "relevancy",
    },
    timeout=30,
)
response.raise_for_status()
news_df = pd.DataFrame(response.json().get("articles", []))
display(news_df.head())
print(f"News rows: {len(news_df):,}")


,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'Yahoo Entertainment'}",Soumya Eswaran,SandiskCorp. (SNDK): AI Boom Fuels Attractive ...,"Parnassus Investments, an investment managemen...",https://finance.yahoo.com/markets/stocks/artic...,https://s.yimg.com/lo/mysterio/api/7032ccc7a40...,2026-09-16T14:56:12Z,"Parnassus Investments, an investment managemen..."
1,"{'id': None, 'name': 'CoinDesk'}",Omkar Godbole,S&P 500 options are the hottest bet on this Hy...,"Hypercall, an exchange built on Hyperliquid, s...",https://www.coindesk.com/markets/2026/10/14/s-...,https://cdn.sanity.io/images/s3y3vcno/producti...,2026-10-14T10:38:47Z,"Trading around the clock is the draw. ""Hyperli..."
2,"{'id': None, 'name': 'CoinDesk'}",James Van Straten,Bitcoin diverges from South Korean stocks as A...,South Korea’s chipmakers lead another sharp se...,https://www.coindesk.com/markets/2026/10/13/bi...,https://cdn.sanity.io/images/s3y3vcno/producti...,2026-10-13T10:38:51Z,South Koreas stock market suffered another sha...
3,"{'id': None, 'name': 'CoinDesk'}","James Van Straten, Omkar Godbole, Francisco Ro...","Bitcoin, ether slip and rebound as South Korea...",Investors are positioning ahead of today's ban...,https://www.coindesk.com/markets/2026/10/13/bi...,https://cdn.sanity.io/images/s3y3vcno/producti...,2026-10-13T11:06:39Z,South Koreas equity benchmark plunged 5.5% on ...
4,"{'id': None, 'name': 'Yahoo Entertainment'}",Soumya Eswaran,SanDisk Corp’s (SNDK) Impact on Small/Mid Cap ...,"Riverwater Partners, an investment management ...",https://finance.yahoo.com/markets/stocks/artic...,https://s.yimg.com/lo/mysterio/api/f1489ae6260...,2026-09-21T12:39:55Z,"Riverwater Partners, an investment management ..."


News rows: 50


In [7]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

market_output = output_dir / f"market_{stock_ticker}.csv"
news_output = output_dir / f"news_{stock_ticker}.json"
stock_df.to_csv(market_output, index=False)
news_df.to_json(news_output, orient="records", date_format="iso")

print("Collection completed.")
print(f"Company: {company_name} ({stock_ticker})")
print(f"Market data: {market_output} ({len(stock_df):,} rows)")
print(f"News data:   {news_output} ({len(news_df):,} rows)")


Collection completed.
Company: Sandisk Corporation (SNDK)
Market data: /content/drive/MyDrive/AAI-520-Project/data/processed/market_SNDK.csv (252 rows)
News data:   /content/drive/MyDrive/AAI-520-Project/data/processed/news_SNDK.json (50 rows)


In [8]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
stock_df.to_csv(output_dir / f"market_{stock_ticker}.csv", index=False)
news_df.to_json(output_dir / f"news_{stock_ticker}.json", orient="records", date_format="iso")
print(f"Saved reusable outputs in repository root: {output_dir}")

Saved reusable outputs in repository root: /content/drive/MyDrive/AAI-520-Project/data/processed
